# LC 1046 — Last Stone Weight
**Difficulty:** Easy &nbsp;|&nbsp; **Category:** Heap
**Pattern:** Max-Heap Simulation — Greedy Smash

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Always smash the two
heaviest stones. Use a max-heap to extract the two
largest in O(log n) each round. If they differ,
push the difference back. Repeat until at most one
stone remains.
</div>

## Official Problem Statement

You are given an array of integers `stones` where
`stones[i]` is the weight of the `i`th stone.

We are playing a game with the stones. On each
turn, we smash together the **two heaviest** stones.
Suppose the heaviest two stones have weights `x`
and `y` with `x <= y`. The result of this smash is:

- If `x == y`, both stones are destroyed.
- If `x != y`, the stone of weight `x` is
  destroyed, and the stone of weight `y` has new
  weight `y - x`.

At the end of the game, there is **at most one**
stone left. Return the weight of the last remaining
stone. If there are no stones left, return `0`.

**Example 1:**
```
Input:  stones = [2,7,4,1,8,1]
Output: 1
```
**Example 2:**
```
Input:  stones = [1]
Output: 1
```

**Constraints:**
- `1 <= stones.length <= 30`
- `1 <= stones[i] <= 1000`

## What This Is Actually Asking

You have a pile of stones. Each round, grab the
two heaviest ones and smash them together. Equal
weights cancel out completely. Unequal weights
leave a fragment equal to the difference.
Keep smashing until one or zero stones are left.
Return the weight of the survivor, or 0.

## Walk Through an Example by Hand

```
stones = [2, 7, 4, 1, 8, 1]

Build max-heap (negate for Python's min-heap):
  heap = [-8,-7,-4,-2,-1,-1]

Round 1:
  pop -8 (y=8)  pop -7 (x=7)
  y != x -> push -(8-7) = -1
  heap = [-4,-2,-1,-1,-1]

Round 2:
  pop -4 (y=4)  pop -2 (x=2)
  y != x -> push -(4-2) = -2
  heap = [-2,-1,-1,-1]

Round 3:
  pop -2 (y=2)  pop -1 (x=1)
  y != x -> push -(2-1) = -1
  heap = [-1,-1,-1]

Round 4:
  pop -1 (y=1)  pop -1 (x=1)
  y == x -> both destroyed
  heap = [-1]

One stone left: -(-1) = 1
Answer: 1
```

## The Picture

```
stones = [2, 7, 4, 1, 8, 1]

Think of a sorting machine that always grabs
the two tallest bars and grinds them down:

  8  7                        <- smash -> 1
  4  2  1  1   +1             <- smash 4,2 -> 2
  2  1  1  1   +1             <- smash 2,1 -> 1
  1  1  1  1                  <- smash 1,1 -> 0
  1            <- survivor

Python's heapq is a MIN-heap. To get a max-heap:
  Store NEGATIVE weights.
  heappop gives the most negative = the largest.

  push  8  ->  push -8
  pop max  ->  pop min, then negate

Each round:
  y = -heappop(heap)   # biggest
  x = -heappop(heap)   # second biggest
  if y != x: heappush(heap, -(y - x))
```

## When To Use This Pattern

- When you need the **two largest values repeatedly**,
  think **max-heap simulation**
- When Python's `heapq` is min-heap only, think
  **negate values to simulate a max-heap**
- When equal elements cancel and unequal leave a
  remainder, think **pop two, push difference**
- When the loop ends with zero or one item, think
  **return 0 if empty, else -heap[0]**

## The Approach

Negate all stone weights and push them into a
min-heap (this simulates a max-heap).
While more than one stone remains, pop the two
largest. If they differ, push the negated difference
back.
When the loop ends, return the last stone's weight
(negate it back), or 0 if the heap is empty.

In [ ]:
import heapq  # Python min-heap — negate for max-heap behaviour
from typing import List

In [ ]:
def test_harness(func):
    tests = [
        # (stones, expected)
        ([2,7,4,1,8,1],  1),
        ([1],             1),
        ([2,2],           0),   # equal — both destroyed
        ([3,3,3],         3),   # two cancel, one left
        ([1,3],           2),
        ([10,4,2,10],     2),
        ([1,1,1,1,1,1],   0),   # all cancel
        ([5],             5),   # single stone
        ([2,3],           1),
        ([1000,999],      1),   # large values
    ]

    passed = 0
    for i, (stones, expected) in enumerate(tests):
        result = func(stones[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"stones={stones} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def lastStoneWeight(stones: List[int]) -> int:
    """
    Return weight of last surviving stone, or 0.

    Negate all weights and heapify (simulates max-heap).
    While heap has 2+ items: pop y and x (largest two).
    If different, push -(y-x) back. Return -heap[0] if
    one stone remains, else 0.

    Time:  O(n log n) — at most n rounds, each O(log n)
    Space: O(n) — heap holds all stones initially
    """
    pass


# Quick debug — run this cell while building
print(lastStoneWeight([2,7,4,1,8,1]))  # 1
print(lastStoneWeight([1]))             # 1
print(lastStoneWeight([2,2]))           # 0
print(lastStoneWeight([3,3,3]))         # 3

In [ ]:
# Uncomment and run when solution is ready
# test_harness(lastStoneWeight)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Sort every round | O(n² log n) | O(1) |
| Max-heap simulation | O(n log n) | O(n) |

The heap pays a one-time O(n) heapify cost, then
each round costs O(log n) — far cheaper than
re-sorting after every smash.

## Real World Connection

At Citi, alert deduplication merges the two highest-
priority unacknowledged alerts on each cycle: if
they share the same root cause they cancel; if they
differ a new composite alert is raised with the
priority delta.
This is Last Stone Weight mapped to alerts — the
max-heap always surfaces the two most urgent events
in O(log n) per merge cycle.
On AWS Step Functions, task retry priorities are
managed the same way: the scheduler pops the two
highest-weight pending tasks and merges or escalates
them without re-sorting the full queue.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra